In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from datasets import Dataset

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)

import torch



In [2]:
df = pd.read_csv(
    r"D:\contract-intelligence-ai\data\processed\final_training_dataset.csv"
)

print(df.shape)

df.head()

(1654, 4)


,contract_id,label,text,category
0,1.0,Enrollment in the Affiliate Program; Restricte...,1. Enrollment in the Affiliate Program; Restri...,General
1,1.0,Affiliate Responsibilities:,2. Affiliate Responsibilities:\n• Affiliate ca...,General
2,1.0,Referral Fee,3. Referral Fee\nFor each Approved Account (as...,Payment
3,1.0,Approved Account,4. Approved Account\nFor purposes of determini...,General
4,1.0,Links,6. Links\nAffiliate agrees to place Chase’s li...,General


In [3]:
df = df[["text", "category"]]

df = df.dropna()

print(df.shape)

(1654, 2)


In [4]:
label_encoder = LabelEncoder()

df["label"] = label_encoder.fit_transform(
    df["category"]
)

print(label_encoder.classes_)

['Confidentiality' 'Employment' 'General' 'Intellectual_Property' 'Legal'
 'Liability' 'Payment' 'Performance' 'Property' 'Termination']


In [5]:
import pickle

with open(
    "label_encoder.pkl",
    "wb"
) as f:

    pickle.dump(
        label_encoder,
        f
    )

In [6]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print(train_df.shape)
print(test_df.shape)

(1323, 3)
(331, 3)


In [7]:
train_dataset = Dataset.from_pandas(
    train_df
)

test_dataset = Dataset.from_pandas(
    test_df
)

In [8]:
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

In [9]:
def tokenize(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

In [10]:
train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/1323 [00:00<?, ? examples/s]

Map:   0%|          | 0/331 [00:00<?, ? examples/s]

In [11]:
train_dataset = train_dataset.rename_column(
    "label",
    "labels"
)

test_dataset = test_dataset.rename_column(
    "label",
    "labels"
)

train_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

test_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

In [12]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(
        label_encoder.classes_
    )
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=3,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    save_strategy="no",

    report_to="none",

    logging_steps=10
)

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [15]:
trainer.train()

c:\Users\SHREEJA\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,1.890200
20,1.536600
30,1.233500
40,1.381300
50,1.687100
60,1.092300
70,1.377800
80,1.601400
90,1.534400
100,1.251700


TrainOutput(global_step=498, training_loss=0.8318314494856869, metrics={'train_runtime': 2833.0102, 'train_samples_per_second': 1.401, 'train_steps_per_second': 0.176, 'total_flos': 262919057587200.0, 'train_loss': 0.8318314494856869, 'epoch': 3.0})